# 00 · Foundations — Part 1: State, Nodes and Edges
### LangGraph · the first notebook

A LangChain chain runs its steps in a straight line. Real workflows are messier: they **branch**, they
**loop**, they **wait for a human**, and they **remember**. LangGraph describes a workflow as a *graph*, and
that one idea is what makes all of those possible.

Picture a shared whiteboard in a support office:

| In the office | In LangGraph | What it is in code |
|---|---|---|
| The **whiteboard** everyone reads | **State** | A dictionary-like object with named fields |
| A **worker** who reads the board and writes what *they* changed | **Node** | A plain Python function |
| The **arrows** saying who works next | **Edges** | `add_edge("a", "b")` |
| Checking the wiring, then opening for business | **Compile** | `builder.compile()` |

That's the whole model. By the end of this notebook you'll have built a working **support-ticket intake
pipeline** from those four pieces.

> 📌 **Checked while building this notebook** (`langgraph 1.2.17`, Python 3.10+; every point below was
> observed by running it)
> - **A node must *return* its changes.** Modifying the state in place does nothing; the change is lost.
> - **A misspelled key in a node's return is silently dropped.** No error, the value just vanishes.
> - **`invoke` returns a plain dictionary.**
> - **Compile catches some wiring mistakes and misses others.** A missing `START` edge or an edge to an
>   undefined node raises `ValueError`. A node that nothing connects to, or a missing edge to `END`, compiles
>   without complaint.
> - **When two branches feed one node, separate `add_edge` calls can make that node run twice**, if the
>   branches are different lengths. The list form `add_edge([...], "node")` waits for all of them (Section 6).
> - **Older tutorials use APIs that have moved:** `MessageGraph` and `config_schema` are deprecated since
>   LangGraph 1.0 (use `StateGraph` with a `messages` key, and `context_schema`), and `create_react_agent`
>   is deprecated in favour of `langchain.agents.create_agent`.
> - Everything here runs **offline, with no API key**.

## Setup

In [ ]:
# %pip install -U langgraph
# %pip install grandalf     # optional: only for drawing a graph as text art (Section 4)


In [ ]:
import re
from datetime import datetime, timedelta
from typing import TypedDict
import importlib.metadata as md

from langgraph.graph import StateGraph, START, END

print("langgraph", md.version("langgraph"))


---
## 1. Your first graph

We'll start with the smallest useful pipeline: take a customer's raw message, tidy it, and decide which team
should get it.

### Step 1 — Declare the state

The state is the whiteboard: it lists every field the workers may read or write. A `TypedDict` is just a
dictionary where we've written down the allowed keys and their types.

In [ ]:
class TicketState(TypedDict):
    raw_text: str     # what the customer typed
    text: str         # the cleaned-up version
    queue: str        # which team gets it


### Step 2 — Write the nodes

A node is a normal function. It receives the **whole state** and returns a dictionary containing **only the
fields it changed**. LangGraph merges that into the state for you.

In [ ]:
def clean(state: TicketState) -> dict:
    """Collapse stray whitespace so later steps see tidy text."""
    return {"text": " ".join(state["raw_text"].split())}


def route(state: TicketState) -> dict:
    """Pick a queue from a few keywords (a real system would use a model)."""
    text = state["text"].lower()
    if any(word in text for word in ("charge", "refund", "invoice", "payment")):
        return {"queue": "billing"}
    return {"queue": "general"}


### Step 3 — Wire it up and compile

`StateGraph(TicketState)` starts a *builder*. We add the nodes, then draw the arrows. `START` and `END` are
built-in markers for "the beginning" and "the end". `compile()` checks the wiring and returns an app you can
run.

In [ ]:
builder = StateGraph(TicketState)

builder.add_node("clean", clean)
builder.add_node("route", route)

builder.add_edge(START, "clean")
builder.add_edge("clean", "route")
builder.add_edge("route", END)

graph = builder.compile()

result = graph.invoke({"raw_text": "   My  payment   was charged twice!   "})
print(result)


Three things to notice:

1. We only supplied `raw_text`. The other fields (`text`, `queue`) don't exist until a node writes them. You
   only need to provide what you know at the start.
2. `invoke` returns the **final state** as a plain dictionary, including the fields we started with.
3. `clean` returned only `{"text": ...}`, yet `raw_text` is still in the result. A node's return value is
   **merged into** the state; it doesn't replace it.

---
## 2. Watch it run

`invoke` only gives you the end result. `stream` lets you watch the work happen, one step at a time. There are
two views worth knowing:

- `stream_mode="updates"` yields **what each node returned**.
- `stream_mode="values"` yields **the whole state** after each step (starting with the initial input).

In [ ]:
sample = {"raw_text": "   My  payment   was charged twice!   "}

print("--- updates: what each node wrote ---")
for update in graph.stream(sample, stream_mode="updates"):
    print(update)

print("\n--- values: the whole state after each step ---")
for state in graph.stream(sample, stream_mode="values"):
    print(state)


---
## 3. A shortcut for straight lines: `add_sequence`

When nodes simply run one after another, `add_sequence` adds them *and* the arrows between them. It takes the
node names from the function names. You still connect the ends to `START` and `END` yourself.

In [ ]:
builder = StateGraph(TicketState)
builder.add_sequence([clean, route])          # adds both nodes and the arrow clean -> route
builder.add_edge(START, "clean")
builder.add_edge("route", END)

same_graph = builder.compile()
print(same_graph.invoke(sample) == graph.invoke(sample), "<- identical to the version we wired by hand")


---
## 4. Reading a graph

A compiled graph can describe itself, which is the fastest way to confirm it is wired the way you meant.

In [ ]:
drawing = graph.get_graph()

print("nodes:", list(drawing.nodes))
print("edges:", [(edge.source, edge.target) for edge in drawing.edges])
print()
print(drawing.draw_mermaid())


That text is **Mermaid**, a small diagram language. Paste it into any Mermaid viewer (for example
mermaid.live) to see the picture. If you prefer text art right in the notebook, `print_ascii()` draws one,
but it needs the optional `grandalf` package:

In [ ]:
try:
    graph.get_graph().print_ascii()
except ImportError:
    print("Install grandalf (pip install grandalf) to draw text art. The Mermaid text above always works.")


---
## 5. Working in parallel: super-steps

So far each node waited for the one before it. But some work is independent. Detecting the **language** of a
ticket and judging its **priority** don't need each other; both just need the cleaned text.

LangGraph runs a graph in **super-steps**. In each super-step, every node that is *ready* runs at the same
time; their updates are applied together; then the next super-step begins.

Here the graph fans out after `clean` and joins back at `finish`:

```
 START → clean ─┬→ detect_language ─┐
                └→ detect_priority ─┴→ finish → END
```

(The two parallel nodes write to **different** fields, so they can't collide. What happens when two nodes write
the *same* field is the next notebook.)

In [ ]:
class IntakeState(TypedDict):
    raw_text: str
    text: str
    language: str
    priority: str
    summary: str


def clean(state: IntakeState) -> dict:
    return {"text": " ".join(state["raw_text"].split())}


def detect_language(state: IntakeState) -> dict:
    has_devanagari = any("\u0900" <= ch <= "\u097f" for ch in state["text"])
    return {"language": "hi" if has_devanagari else "en"}


def detect_priority(state: IntakeState) -> dict:
    text = state["text"].lower()
    return {"priority": "high" if any(w in text for w in ("urgent", "outage", "fraud")) else "normal"}


def finish(state: IntakeState) -> dict:
    return {"summary": f"{state['priority']} priority, {state['language']}"}


builder = StateGraph(IntakeState)
for name, fn in [("clean", clean), ("detect_language", detect_language),
                 ("detect_priority", detect_priority), ("finish", finish)]:
    builder.add_node(name, fn)

builder.add_edge(START, "clean")
builder.add_edge("clean", "detect_language")
builder.add_edge("clean", "detect_priority")
builder.add_edge(["detect_language", "detect_priority"], "finish")     # the join: wait for BOTH
builder.add_edge("finish", END)

intake = builder.compile()

print("Each line below is ONE super-step (the whole state after it):\n")
for step, state in enumerate(intake.stream({"raw_text": "URGENT: site is in an outage"}, stream_mode="values")):
    print(f"step {step}: {state}")


Read the output top to bottom: step 1 is `clean`; **step 2 contains both `language` and `priority`**, because
those two nodes ran in the same super-step; step 3 is `finish`.

---
## 6. Joining branches — the trap that runs a node twice

The join above used the **list form**: `add_edge(["detect_language", "detect_priority"], "finish")`. That means
"run `finish` only when *both* have finished." It looks like an unimportant detail. It isn't.

Suppose one branch takes **two** steps (priority detection is followed by a VIP check) while the other takes
one. If you wire the join with **separate** `add_edge` calls, `route` is triggered each time a branch arrives,
so it runs **twice**. Imagine that node sends an email.

In [ ]:
class JoinState(TypedDict):
    text: str
    language: str
    priority: str
    vip: str
    routed: str

route_runs = []          # a side-channel just to COUNT how often route() runs


def language_step(state: JoinState) -> dict:
    return {"language": "en"}

def priority_step(state: JoinState) -> dict:
    return {"priority": "high"}

def vip_check(state: JoinState) -> dict:
    return {"vip": "no"}                         # this branch is one step LONGER

def route_ticket(state: JoinState) -> dict:
    route_runs.append(1)
    return {"routed": "done"}


def build(join_with_list: bool):
    b = StateGraph(JoinState)
    for name, fn in [("language_step", language_step), ("priority_step", priority_step),
                     ("vip_check", vip_check), ("route_ticket", route_ticket)]:
        b.add_node(name, fn)
    b.add_edge(START, "language_step")
    b.add_edge(START, "priority_step")
    b.add_edge("priority_step", "vip_check")
    if join_with_list:
        b.add_edge(["language_step", "vip_check"], "route_ticket")          # wait for both
    else:
        b.add_edge("language_step", "route_ticket")                          # two separate arrows...
        b.add_edge("vip_check", "route_ticket")
    b.add_edge("route_ticket", END)
    return b.compile()


route_runs.clear(); build(join_with_list=False).invoke({"text": "hi"})
print("separate add_edge calls  -> route_ticket ran", len(route_runs), "time(s)")

route_runs.clear(); build(join_with_list=True).invoke({"text": "hi"})
print("add_edge([...], 'node')  -> route_ticket ran", len(route_runs), "time(s)")


**Rule of thumb:** whenever a node must wait for *several* predecessors, use the list form. With branches of
equal length the two forms happen to behave the same (we measured that too), which is exactly why the bug hides
until someone adds a step to one branch.

---
## 7. Common mistakes, on purpose

Each of these is something you will do at least once. Seeing them fail here is cheaper than finding them in a
bigger graph.

### Mistake 1 — changing the state instead of returning an update

In [ ]:
class Counter(TypedDict):
    count: int


def sneaky(state: Counter) -> dict:
    state["count"] = 99          # modifies the local copy...
    return {}                    # ...and returns no update

b = StateGraph(Counter); b.add_node("sneaky", sneaky); b.add_edge(START, "sneaky"); b.add_edge("sneaky", END)
print(b.compile().invoke({"count": 0}), " <- still 0: the in-place change was lost")


### Mistake 2 — misspelling a key (silent)

In [ ]:
def typo(state: Counter) -> dict:
    return {"cuont": 5}          # note the typo

b = StateGraph(Counter); b.add_node("typo", typo); b.add_edge(START, "typo"); b.add_edge("typo", END)
print(b.compile().invoke({"count": 0}), " <- no error, and the 5 simply vanished")


### Mistake 3 — returning something that isn't a dictionary

In [ ]:
def returns_text(state: Counter):
    return "done"

b = StateGraph(Counter); b.add_node("n", returns_text); b.add_edge(START, "n"); b.add_edge("n", END)
try:
    b.compile().invoke({"count": 0})
except Exception as error:
    print(type(error).__name__, "->", str(error).splitlines()[0])


### Mistakes 4 and 5 — wiring errors that compile *catches*

In [ ]:
def keep(state: Counter) -> dict:
    return {}

def try_compile(label, build_graph):
    try:
        build_graph().compile()
        print(f"{label}: compiled with no error")
    except Exception as error:
        print(f"{label}: {type(error).__name__} -> {error}")

def forgot_start():
    b = StateGraph(Counter); b.add_node("a", keep); b.add_edge("a", END); return b

def edge_to_nowhere():
    b = StateGraph(Counter); b.add_node("a", keep); b.add_edge(START, "a"); b.add_edge("a", "ghost"); return b

try_compile("no edge from START      ", forgot_start)
try_compile("edge to an unknown node ", edge_to_nowhere)


### Mistakes 6 and 7 — wiring errors that compile *misses*

These two build without any complaint, so you only find out by running the graph.

In [ ]:
def with_orphan():
    b = StateGraph(Counter)
    b.add_node("a", lambda s: {"count": s["count"] + 1})
    b.add_node("island", lambda s: {"count": s["count"] + 100})      # nothing points at it
    b.add_edge(START, "a"); b.add_edge("a", END)
    return b

def no_end_edge():
    b = StateGraph(Counter)
    b.add_node("a", lambda s: {"count": s["count"] + 1})
    b.add_edge(START, "a")                                             # no arrow to END
    return b

try_compile("a node nothing connects to", with_orphan)
print("   ...and running it:", with_orphan().compile().invoke({"count": 0}), " <- 'island' never ran (count is 1, not 101)")

try_compile("no edge to END            ", no_end_edge)
print("   ...and running it:", no_end_edge().compile().invoke({"count": 0}), " <- fine: the run just stops when nothing is left to do")


A good habit: after building a graph, print `get_graph().draw_mermaid()` (Section 4) and **look** at it. An
orphan node shows up as a box with no arrows.

---
## Real-world capstone: a support-ticket intake pipeline

Every support desk starts the same way: a message arrives, and something must work out **what it is**, **who
should get it**, and **how soon it must be answered**. We'll build that with everything above, with no AI model
needed.

```
 START → clean ─┬→ detect_language ─┐
                ├→ detect_priority ─┼→ finalize → END
                └→ assign_queue ────┘
```

Three independent checks run in the same super-step, then one node joins them (with the **list form**) to work
out the response deadline.

In [ ]:
class Intake(TypedDict):
    raw_text: str
    received_at: str          # ISO time the ticket arrived, passed in so results are repeatable
    text: str
    language: str
    priority: str
    queue: str
    due_at: str
    needs_translation: bool


SLA_HOURS = {"high": 4, "medium": 24, "low": 72}


def clean(state: Intake) -> dict:
    return {"text": " ".join(state["raw_text"].split())}


def detect_language(state: Intake) -> dict:
    devanagari = any("\u0900" <= ch <= "\u097f" for ch in state["text"])
    return {"language": "hi" if devanagari else "en"}


def detect_priority(state: Intake) -> dict:
    text = state["text"].lower()
    if any(w in text for w in ("urgent", "outage", "not working", "charged twice", "fraud")):
        return {"priority": "high"}
    if any(w in text for w in ("refund", "error", "failed", "crash")):
        return {"priority": "medium"}
    return {"priority": "low"}


def assign_queue(state: Intake) -> dict:
    text = state["text"].lower()
    if any(w in text for w in ("charge", "refund", "invoice", "payment")):
        return {"queue": "billing"}
    if any(w in text for w in ("crash", "error", "bug", "outage", "not working")):
        return {"queue": "technical"}
    if any(w in text for w in ("login", "password", "account", "email", "पासवर्ड")):
        return {"queue": "account"}
    return {"queue": "general"}


def finalize(state: Intake) -> dict:
    due = datetime.fromisoformat(state["received_at"]) + timedelta(hours=SLA_HOURS[state["priority"]])
    return {"due_at": due.isoformat(timespec="minutes"), "needs_translation": state["language"] != "en"}


builder = StateGraph(Intake)
for name, fn in [("clean", clean), ("detect_language", detect_language), ("detect_priority", detect_priority),
                 ("assign_queue", assign_queue), ("finalize", finalize)]:
    builder.add_node(name, fn)

builder.add_edge(START, "clean")
for branch in ("detect_language", "detect_priority", "assign_queue"):
    builder.add_edge("clean", branch)
builder.add_edge(["detect_language", "detect_priority", "assign_queue"], "finalize")      # wait for all three
builder.add_edge("finalize", END)

intake_pipeline = builder.compile()
print(intake_pipeline.get_graph().draw_mermaid())


In [ ]:
tickets = [
    "URGENT:   my payment was charged twice, please refund",
    "The app crashes when I open Settings",
    "How do I change my email address?",
    "मेरा पासवर्ड रीसेट नहीं हो रहा",
]

print(f"{'ticket':52} {'lang':4} {'priority':8} {'queue':10} {'due':16} translate?")
results = []
for text in tickets:
    r = intake_pipeline.invoke({"raw_text": text, "received_at": "2026-10-30T09:00:00"})
    results.append(r)
    print(f"{r['text'][:50]:52} {r['language']:4} {r['priority']:8} {r['queue']:10} {r['due_at']:16} {r['needs_translation']}")


In [ ]:
# The repeatable parts of the lesson, pinned down so they can't silently drift:
expected = [
    ("en", "high",   "billing",   "2026-10-30T13:00", False),
    ("en", "medium", "technical", "2026-10-31T09:00", False),
    ("en", "low",    "account",   "2026-11-02T09:00", False),
    ("hi", "low",    "account",   "2026-11-02T09:00", True),
]
actual = [(r["language"], r["priority"], r["queue"], r["due_at"], r["needs_translation"]) for r in results]
assert actual == expected, actual
print("All four tickets routed as expected.")


Look at what you built: five tiny functions, each doing one job and **knowing nothing about the others**,
plus a few lines of wiring that decide the order. Swapping the keyword rule in `detect_priority` for a language
model later would change one function and nothing else. That separation is why graphs scale.

### Try it yourself
1. Add a fourth parallel check, `detect_vip`, that sets a `vip: bool` field if the text mentions "premium".
   Remember to add the field to `Intake`, a node, an edge from `clean`, and to the **list** in the join.
2. Introduce each mistake from Section 7 into the capstone, one at a time, and see how loudly (or quietly)
   each one fails.
3. Add a second node after `detect_priority` (say `check_escalation`) and wire the join with *separate*
   `add_edge` calls. Count how often `finalize` runs, then fix it with the list form.
4. Print `intake_pipeline.get_graph().draw_mermaid()` and paste it into a Mermaid viewer. Does the picture
   match the diagram above?
5. Run `intake_pipeline.stream(..., stream_mode="updates")` on one ticket. Which three updates arrive together?

---
**Next:** `02_reducers_and_messages.ipynb` — what happens when two nodes write the *same* field at the same
time, and how LangGraph keeps a conversation's messages.
